In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import anderson, kstest
from scipy.stats import t as student_t
from statsmodels.tsa.stattools import adfuller, kpss, acf
import matplotlib.pyplot as plt
from statsmodels.stats.diagnostic import acorr_ljungbox

In [ ]:
df_raw = pd.read_parquet("data/processed/crypto.parquet")
df_raw.head()

In [ ]:
df_pct_chg = df_raw.pivot(index='close_time', columns='asset', values='log_return')
print(df_pct_chg.isna().sum())
df_pct_chg.tail()

In [ ]:
stats = []
for asset in df_pct_chg.columns:
    s = df_pct_chg[asset].dropna()
    if s.empty:
        continue

    stats.append({
        "asset": asset,
        "first_date": s.index.min(),
        "last_date": s.index.max(),
        "n_returns": s.count(),
        "annualized_mean_return": s.mean()*252*24*4,
        "annualized_std_return": s.std()*np.sqrt(252*24*4),
        "skewness": s.skew(),
        "kurtosis": s.kurt(),
        "min_return": s.min(),
        "max_return": s.max()
    })

stats_df = pd.DataFrame(stats).sort_values("first_date")

stats_df.T

In [ ]:
summary_rows = []
trimmed_rows = []

for asset in df_pct_chg.columns:
    s = df_pct_chg[asset].dropna()
    if s.empty:
        continue

    mean_val = s.mean()
    std_val = s.std()
    lower = mean_val - 1.96 * std_val
    upper = mean_val + 1.96 * std_val

    within_ci = s[(s >= lower) & (s <= upper)].copy()
    dropped = s.size - within_ci.size

    summary_rows.append({
        "asset": asset,
        "n_values_considered": within_ci.size,
        "n_values_dropped": dropped,
        "annualized_mean": within_ci.mean()*252*24*4,
        "annualized_std": within_ci.std()*np.sqrt(252*24*4),
        "skewness": within_ci.skew(),
        "kurtosis": within_ci.kurt(),
        "min": within_ci.min(),
        "max": within_ci.max(),
    })

    for value in within_ci:
        trimmed_rows.append({
            "asset": asset,
            "trimmed_value": value,
        })

summary_df = pd.DataFrame(summary_rows)
trimmed_values_df = pd.DataFrame(trimmed_rows)

summary_df.T

In [ ]:
returns_df = df_pct_chg.copy().dropna(how='all')
squared_returns_df = returns_df.pow(2)

returns_df.head()
squared_returns_df.head()

In [ ]:
assets = list(returns_df.columns)
assets_to_plot = assets[:8]

fig, axes = plt.subplots(4, 2, figsize=(16, 20), sharex=True)
axes = axes.flatten()

for ax, asset in zip(axes, assets_to_plot):
    ax.plot(returns_df.index, returns_df[asset], color='tab:blue', linewidth=0.8)
    ax.set_title(f'{asset} - Returns')
    ax.set_ylabel('Return')
    ax.tick_params(axis='x', rotation=45)

for ax, asset in zip(axes[len(assets_to_plot):], assets_to_plot):
    ax.plot(squared_returns_df.index, squared_returns_df[asset], color='tab:red', linewidth=0.8)
    ax.set_title(f'{asset} - Squared Returns')
    ax.set_ylabel('Squared Return')
    ax.tick_params(axis='x', rotation=45)

for ax in axes[len(assets_to_plot)*2:]:
    ax.axis('off')

fig.suptitle('Returns and Squared Returns by Asset', fontsize=14)
plt.subplots_adjust(wspace=0.25, hspace=0.45)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

In [ ]:
rows = []
for asset in df_pct_chg.columns:
    s = df_pct_chg[asset].dropna()
    if s.empty:
        continue

    for test_name, regression, label in [
        ("ADF", "c", "without trend"),
        ("ADF", "ct", "with trend"),
        ("PP", "c", "without trend"),
        ("PP", "ct", "with trend"),
    ]:
        if test_name == "ADF":
            result = adfuller(s, autolag="AIC", regression=regression)
            statistic = result[0]
            pvalue = result[1]
            used_lag = result[2]
            # nobs = result[3]
            # critical_values = result[4]
        else:
            result = kpss(s, regression=regression, nlags="auto")
            statistic = result[0]
            pvalue = result[1]
            used_lag = result[2]
            # nobs = result[3]
            # critical_values = result[4]

        rows.append({
            "asset": asset,
            "test": test_name,
            "specification": label,
            "statistic": statistic,
            "pvalue": pvalue,
            "used_lag": used_lag,
            # "nobs": nobs,
            # "critical_values": critical_values,
            # "reject_5pct": pvalue < 0.05,
        })

unit_root_results_df = pd.DataFrame(rows)
print(unit_root_results_df)

In [ ]:
dist_names = ["norm", "expon", "logistic", "gumbel_l", "gumbel_r"]

results = []
for asset in df_pct_chg.columns:
    pct = df_pct_chg[asset].dropna()
    if pct.empty:
        continue

    for dist in dist_names:
        stat, critical_values, significance_levels = anderson(pct, dist=dist)
        results.append({
            "asset": asset,
            "distribution": dist,
            "method": "anderson",
            "statistic": stat,
            "critical_values": critical_values,
            "significance_levels": significance_levels,
            "reject_5pct": stat > critical_values[list(significance_levels).index(5)],
        })

    params = student_t.fit(pct)
    ks_stat, ks_pvalue = kstest(pct, "t", args=params)
    results.append({
        "asset": asset,
        "distribution": "student_t",
        "method": "kstest",
        "statistic": ks_stat,
        "params": params,
        "reject_5pct": ks_pvalue > 0.05,
        "pvalue": ks_pvalue,
    })

results_df = pd.DataFrame(results)
print(results_df[["asset", "distribution", "method", "statistic", "reject_5pct", "pvalue"]])

In [ ]:
lags = [10, 15, 20]
rows = []

for asset in df_pct_chg.columns:
    s = squared_returns_df[asset].dropna()
    if s.empty:
        continue

    for lag in lags:
        lb = acorr_ljungbox(s, lags=[lag], return_df=True)
        row = lb.loc[lag]
        rows.append({
            "asset": asset,
            "lag": lag,
            "lb_statistic": row["lb_stat"],
            "lb_pvalue": row["lb_pvalue"],
            "reject_5pct": row["lb_pvalue"] < 0.05,
        })

ljung_box_results_df = pd.DataFrame(rows)
print(ljung_box_results_df)

In [ ]:
max_lag = 100

for asset in returns_df.columns:
    s = returns_df[asset].dropna()
    if s.empty:
        continue

    s_abs = s.abs()
    s_sq = s.pow(2)
    n = s.size

    acf_ret = acf(s, nlags=max_lag, fft=True)
    acf_abs = acf(s_abs, nlags=max_lag, fft=True)
    acf_sq = acf(s_sq, nlags=max_lag, fft=True)

    ci = 1.96 / np.sqrt(n)

    fig, ax = plt.subplots(figsize=(8, 4))
    l1, = ax.plot(acf_ret, label='retc', color='tab:blue')
    l2, = ax.plot(acf_abs, label='abs(retc)', color='tab:orange')
    l3, = ax.plot(acf_sq, label='retc^2', color='tab:green')

    # ax.axhline(ci, color='purple', linestyle='--', linewidth=0.9, label='cip')
    # ax.axhline(-ci, color='red', linestyle='--', linewidth=0.9, label='cin')

    ax.set_xlim(0, max_lag)
    ax.set_xlabel('Lag')
    ax.set_ylabel('ACF')
    ax.set_title(f'{asset}')
    ax.legend(loc='upper right')
    plt.tight_layout()

    # show and save
    plt.show()
    # try:
    #     fig.savefig(f"acf_{asset}.png", dpi=150, bbox_inches='tight')
    # except Exception:
    #     pass
    # plt.close(fig)

In [ ]:
# # Faster unit-root test run: test a subset of the data first and keep the same output structure.
# # You can increase the sample size later if needed.

# sample_size = 10000
# assets_to_test = list(df_pct_chg.columns)

# rows = []
# for asset in assets_to_test:
#     s = df_pct_chg[asset].dropna()
#     if s.empty:
#         continue

#     # Use the most recent observations to reduce runtime.
#     s = s.tail(sample_size)

#     for test_name, regression, label in [
#         ("ADF", "c", "without trend"),
#         ("ADF", "ct", "with trend"),
#     ]:
#         result = adfuller(s, autolag="AIC", regression=regression)
#         rows.append({
#             "asset": asset,
#             "test": test_name,
#             "specification": label,
#             "statistic": result[0],
#             "pvalue": result[1],
#             "used_lag": result[2],
#             "nobs": result[3],
#             "critical_values": result[4],
#             "reject_5pct": result[1] < 0.05,
#         })

# unit_root_results_df = pd.DataFrame(rows)
# print(unit_root_results_df)